## Hyperparameters in ML Models:

### What Are Hyperparameters

- Value that determines part of the learning process and is not affected by training unlike a parameter, which is learned during the model training process.

- **k-nearest neighbors** algorithm, k is a hyperparameter. k determines how many neighbors will be used, and training data does not change k.

- **Decision trees** also use hyperparameters. Before training a decision tree classifier, you might want to specify how deep the tree can go (i.e., how many splits are allowed before arriving at a leaf). You might also want to specify the minimum number of samples that are present in a node in order to split that node. Both of those values are hyperparameters. They determine the structure of the model, but they are not learned during training.

- **Regularization** factors are another example. In linear or logistic regression, regularization is a term that is added to the loss function in order to penalize models with large coefficients. The coefficients are the parameters of the model here! The regularization factor is a hyperparameter. The value of the regularization factor affects how large the coefficients of a regression model will be, but the regularization factor is independent of training data.

### Hyperparameter Tuning

- `bias-variance tradeoff` : balance between overfitting anf underfitting

    - `bias` : the difference between a model’s predictions and the correct values. Models with a lot of bias underfit the data and will perform poorly on both testing and training data. In biased models, the structure of the model overpowers the training data

    - `variance` : the dependence of a model on training data. A model has high variance if different training data results in widely varying outcomes. This often leads to overfitting a model to training data. Overfit, high-variance models generally perform well on training data, but poorly on testing data. In this case, the training data overpowers the structure of the model.

    - *The tradeoff is by decreasing the bias usually the variance is increased and conversely, decreasing variance usually increases bias*

    - In the case of supervised machine learning, we want hyperparameters that make a good compromise between bias and variance.

    ---

    ### Hyperparameter Tuning Methods

    - `grid search` algorithm : works by tuning a model on predetermined list of hyperparameter values. Tries every hyperparameter value on the list and uses the one that makes the model perform best. 

    - `random search` algorithm : values are randomly chosen rather than predetermined. It selects the hyperparameter that performed the best.

    - `bayesian optimization` : iterates through different hyperparameter values, each time the Bayesian optimization algorithm evaluates a new hyperparameter value, it gains more information about where it should look for the best hyperparameter value.

    - `genetic algorithms` : goes through several generations of hyperparameter values. Within each generation, the fittest (i.e., best-performing) hyperparameter values are slightly mutated (i.e., changed) in order to produce the next generation.

> ***When tuning hyperparameters, it’s important to split the data into training, testing, and validation data.*** 

- Training data is used to train the model. 

- Validation data is used to evaluate hyperparameters.

- After a hyperparameter is tuned, the model can be tested on testing data. 

- This data allows for an estimate of model performance that isn’t affected by the hyperparameter tuning process.

![hyperparameter tuning models](images/hyperparameters_models.png)

# Hyperparameter Tuning

> **1. Choose dataset.**

> **2. Choose a classification or regression problem to solve.**

> **3. Choose a machine learning model to solve it with.**

---

> **Evaluating Results:**

- `.best_estimator_` gives us the best estimator

- `.best_score_` gives us the mean cross-validated score corresponding to the best estimator

- `.best_params_` gives us the set of hyperparameters that correspond to the best estimator

- `.cv_results_` gives us the score for each hyprparameter combination in the grid.

## **SETUP: GridSearchCV**

In [ ]:
# SETUP:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# Load the data set
cancer = load_breast_cancer()

# Split into training and testing data
X_train, X_test, y_train, y_test = train_test_split(cancer.data, cancer.target, random_state = 19)

# Intializaing model and dictionary of hyperparameters
lr = LogisticRegression(solver='liblinear', max_iter=1000)
parameters = {'penalty': ['l1', 'l2'], 'C': [1, 10, 100]}

# Setting up Grid Search
clf = GridSearchCV(lr, parameters)

### EVALUATE RESULTS:

In [ ]:
# Fit clf to training data and get best hyperparameters:
best_model = clf.fit(X_train, y_train).best_estimator_

print(best_model)
print(clf.best_params_)

# Calculate training and test scores of the best estimator:
best_score = clf.best_score_
test_score = clf.score(X_test, y_test)

print(best_score)
print(test_score)

# Viewing grid search results:
hyperparameter_grid = pd.DataFrame(clf.cv_results_['params'])
grid_scores = pd.DataFrame(clf.cv_results_['mean_test_score'], columns=['score'])

df = pd.concat([hyperparameter_grid, grid_scores], axis = 1)
print(df)

---

## **SETUP: RandomSearchCV**

- Searches over hyperparameters by drawing a list of random values from distributions. 
    - instead of selecting from predetermined lists, random values would be drawn.

- `RandomSearchCV` requires three arguments:

    - estimator: the machine learning model whose hyperparameters we’re tuning; this is exactly the same for GridSearchCV

    - param_distributions: a dictionary which specifies the hyperparameters as keys and corresponding distributions to draw lists of values from for each hyperparameter. In GridSearchCV, we instead had param_grid, a dictionary representing the grid of hyperparameters to search from

    - n_iter: the number of times the algorithm needs to randomly draw from the distributions. The default value for this is 10.

- specify a probability distribution for each hyperparameter:

```python
from scipy.stats import uniform
distributions = {'penalty': ['l1', 'l2'], 'C': uniform(loc=0, scale=100)}
```

- **discrete uniform distibution** *: every item in the list has an equal chance of being selected.*





In [ ]:
# SETUP
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform

# Load the data set
cancer = load_breast_cancer()

# Split the data into training and testing sets
X = cancer.data
y = cancer.target
X_train, X_test, y_train, y_test = train_test_split(X, y)

# Create distributions to draw hyperparameters from
distributions = {'penalty': ['l1', 'l2'], 'C': uniform(loc=0, scale=100)}

# Check distributions
first_draw = distributions['C'].rvs(10)
second_draw = distributions['C'].rvs(10)

print(first_draw)
print(second_draw)

# Define model and initialize random search
# Logistic regression model
lr = LogisticRegression(solver='liblinear', max_iter=1000)

# RandomSearchCV model
clf = RandomizedSearchCV(lr, distributions, n_iter=8)

### EVALUATE RESULTS:

In [ ]:
# fit clf to training data and get best hyperparameters
best_model = clf.fit(X_train, y_train).best_estimator_

print(best_model)
print(clf.best_params_)

# calculate training and test scores of the best estimator
best_score = clf.best_score_
test_score = clf.score(X_test, y_test)

print(best_score)
print(test_score)

# viewing random search results
hyperparameter_values = pd.DataFrame(clf.cv_results_['params'])
randomsearch_scores = pd.DataFrame(clf.cv_results_['mean_test_score'], columns=['score'])

df= pd.concat([hyperparameter_values, randomsearch_scores], axis=1)
print(df)